<!-- field-paper-role-banner -->
> 🟦 **실습본** — TODO를 직접 구현하고 assertion을 통과시키세요. 정답본은 같은 위치의 필요한 부분만 확인합니다.

# 생성 모델 · 08. Score SDE

## Score-Based Generative Modeling through Stochastic Differential Equations

- **원 논문:** [Score-Based Generative Modeling through Stochastic Differential Equations](https://arxiv.org/abs/2011.13456)
- **저자 / 발표:** Yang Song, Jascha Sohl-Dickstein, Diederik P. Kingma, Abhishek Kumar, Stefano Ermon, and Ben Poole · ICLR (2021)
- **난이도 / 예상 시간:** 고급 · 약 70분
- **선수 지식:** score matching, SDE, Euler discretization
- **로컬 학습 데이터:** `data/field_curriculum/generative_samples.npz`

이 자료는 핵심 메커니즘을 확인하는 **하드웨어 인식 교육용 미니 재현**입니다.
PyTorch 학습은 CUDA → Apple MPS → CPU 순으로 자동 선택하고, NumPy·환경 simulation 같은
CPU 연산은 그대로 유지합니다. 원 논문의 전체 데이터·모델·학습 예산·최종 benchmark를
재현했다는 뜻은 아닙니다.

## 재현 범위

**보존하는 핵심:** 2D 점에서 VP-SDE marginal score를 학습하고 reverse-time Euler sampling을 수행한다.

**축소하거나 생략한 부분:** 논문은 continuous-time score networks와 predictor-corrector samplers로 고해상도 이미지를 생성한다. 축소판은 VP-SDE와 reverse drift를 2D로 구현한다.

완료 기준은 정답과 문자가 같은 코드가 아니라, TODO 구현·불변식 assertion·작은 metric 또는
시각화·마지막 해석을 모두 마치는 것입니다.

## 원 논문 ↔ 실습 지도

        절·수식·그림 번호는 위 primary source를 기준으로 합니다. 판본에 따라 페이지가 달라질 수
        있으므로 절 제목과 식 번호를 함께 확인하세요.

        | 원 논문의 위치 | 대응 코드·실험 | 확인할 증거 |
        |---|---|---|
        | §3.1 Eq. (5); §3.2 Eq. (6) | forward and reverse SDE | forward drift/diffusion과 score가 들어간 reverse drift를 연결한다. |
| §3.3, Eq. (7) | continuous denoising score matching | perturbation kernel의 conditional score를 회귀한다. |
| §3.4, Eq. (11) | VP-SDE marginal과 conditional score target | m(t), σ(t), -z/σ(t)의 shape와 수치 범위를 검증한다. |

## 핵심 재현

        VP-SDE의 marginal $x_t=m(t)x_0+\sigma(t)z$를 이용해 Eq. (7)의 conditional score
        $-z/\sigma(t)$를 회귀합니다. 이후 Eq. (6)의 reverse-time drift를 Euler로 적분합니다.

In [ ]:
from pathlib import Path
import math
import numpy as np
import matplotlib.pyplot as plt
import torch
from torch import nn
import torch.nn.functional as F
from llm_engineering_lab.acceleration import get_accelerator

torch.manual_seed(381)
np.random.seed(381)
ACCELERATOR = get_accelerator()
DEVICE = ACCELERATOR.device
# These teaching batches are intentionally tiny; GPU launch/transfer
# overhead can dominate, so use AI_LAB_DEVICE=cpu for minimum wall time.

DATA_PATH = Path("data/field_curriculum/generative_samples.npz")
assert DATA_PATH.exists(), f"준비된 로컬 데이터가 없습니다: {DATA_PATH}"
raw = np.load(DATA_PATH)
expected = {"points", "point_labels", "images", "image_labels", "domain_a", "domain_b"}
assert expected.issubset(raw.files), f"NPZ 키 불일치: {sorted(raw.files)}"
points = torch.tensor(raw["points"], dtype=torch.float32)
point_labels = torch.tensor(raw["point_labels"], dtype=torch.long)
images = torch.tensor(raw["images"], dtype=torch.float32)
image_labels = torch.tensor(raw["image_labels"], dtype=torch.long)
domain_a = torch.tensor(raw["domain_a"], dtype=torch.float32)
domain_b = torch.tensor(raw["domain_b"], dtype=torch.float32)
points, point_labels, images, image_labels, domain_a, domain_b = ACCELERATOR.move(
    points, point_labels, images, image_labels, domain_a, domain_b
)


# Seeded random draws stay on CPU so CUDA, MPS, and CPU follow the same
# sampling stream; complete tensors are then transferred to the lab device.
def randn_device(size, *, generator):
    return torch.randn(size, generator=generator).to(DEVICE)


def rand_device(size, *, generator):
    return torch.rand(size, generator=generator).to(DEVICE)


def randint_device(high, size, *, generator):
    return torch.randint(high, size, generator=generator).to(DEVICE)


if float(images.min()) < 0.0:
    images_01 = ((images + 1.0) / 2.0).clamp(0.0, 1.0)
else:
    images_01 = images.clamp(0.0, 1.0)
assert points.shape == (600, 2) and images.shape == (300, 1, 8, 8)
assert domain_a.shape == domain_b.shape == (600, 2)
print(ACCELERATOR.summary())
print("local data:", tuple(points.shape), tuple(images.shape), tuple(domain_a.shape))

## 포트폴리오 구현 설계: 수식 → 책임 → 검증

논문의 핵심 목적을 실행 가능한 객체의 `forward`, `loss`, `update`, `evaluate`로
나눕니다. 아래 식이 클래스 내부에서 생략되지 않고 어떤 tensor 연산이 되는지
메서드별로 추적해 보세요.

$$
\mathcal L=\mathbb E_{t,x_0,x_t}\lambda(t)\lVert s_\theta(x_t,t)-\nabla_{x_t}\log p_{0t}(x_t|x_0)\rVert_2^2
$$

- **기호와 역할:** $s_\theta$는 score network, $p_{0t}$는 VP-SDE perturbation kernel, $\lambda(t)=\sigma(t)^2$는 시간별 가중치입니다.
- **shape 계약:** `point xt [N, 2] + time [N, 1] → score [N, 2]`
- **논문 위치:** `§3.1 Eq. (5); §3.2 Eq. (6)`의 **forward and reverse SDE**
- **코드 Task:** VP marginal, conditional score target, weighted loss와 reverse Euler step을 구현합니다.
- **학습·평가 흐름:** continuous t perturb → score update → t=1..0.02 reverse-SDE sampling
- **원문 대비 한계:** 논문은 continuous-time score networks와 predictor-corrector samplers로 고해상도 이미지를 생성한다. 축소판은 VP-SDE와 reverse drift를 2D로 구현한다.
- **구현 이유:** 수식의 tensor 경계와 optimizer 책임을 클래스 메서드별로
  분리해 코드와 논문을 한 줄씩 대조할 수 있게 합니다.
- **완료 증거:** 마지막 셀에서 `update`, parameter 변화, 평가 metric을
  모두 `assert`합니다.

정답 클래스는 교육용 편의 함수가 핵심 계산을 대신하지 않도록 loss와 update 순서를
메서드 본문에 드러냅니다. 실습본에서는 같은 공개 API를 유지한 채 TODO를 채우세요.

In [ ]:
class ScoreSdeLab(nn.Module):
    """Score SDE의 핵심 학습·평가 경로. 입출력 계약: point xt [N, 2] + time [N, 1] → score [N,
    2]."""

    def __init__(self, score_network, optimizer):
        """구성 요소와 optimizer를 저장한다. 학습은 update에서만 수행한다."""
        raise NotImplementedError("TODO: __init__ 본문을 구현하세요.")

    def config(self):
        """재현 범위와 핵심 하이퍼파라미터를 dict로 반환한다."""
        raise NotImplementedError("TODO: config 본문을 구현하세요.")

    def marginal(self, clean, time, noise):
        """VP-SDE 주변분포의 noisy sample, target score, std를 반환한다."""
        raise NotImplementedError("TODO: marginal 본문을 구현하세요.")

    def forward(self, noisy, time):
        """논문 순전파를 계산한다. shape 계약: point xt [N, 2] + time [N, 1] → score [N, 2]."""
        raise NotImplementedError("TODO: forward 본문을 구현하세요.")

    def loss(self, clean, time, noise):
        """논문의 목적 함수를 미분 가능한 scalar tensor로 반환한다."""
        raise NotImplementedError("TODO: loss 본문을 구현하세요.")

    def update(self, clean, time, noise):
        """zero_grad, backward, step을 수행하고 유한한 float loss를 반환한다."""
        raise NotImplementedError("TODO: update 본문을 구현하세요.")

    def reverse_euler_maruyama(self, batch_size, steps, generator):
        """논문 알고리즘의 공개 보조 연산을 수행한다."""
        raise NotImplementedError("TODO: reverse_euler_maruyama 본문을 구현하세요.")

    def evaluate(self, clean, time, noise):
        """no_grad 경로에서 논문과 연결된 정량 지표 dict를 반환한다."""
        raise NotImplementedError("TODO: evaluate 본문을 구현하세요.")

### 단계 2. 핵심 연산 검증

- **논문의 어느 부분인가:** `§3.4, Eq. (11)` — **VP-SDE marginal과 conditional score target**
- **구현 이유:** m(t), σ(t), -z/σ(t)의 shape와 수치 범위를 검증한다.
- **읽을 코드:** 아래 `implementation` 셀에서 중간 tensor의 shape, gradient가 흐르는
  경로와 `detach` 또는 `no_grad` 경계를 확인합니다.

실행 결과만 보지 말고, 이 단계의 입력과 출력이 앞선 수식의 어떤 기호인지 먼저
주석으로 적은 뒤 실습하세요.
- **완료 증거:** 아래 셀의 `assert`와 출력 metric이 shape·loss 계약을 검사합니다.

In [ ]:
def vp_stats(t):
    """정답과 동일한 공개 signature를 유지한다. shape 계약: point xt [N, 2] + time [N, 1] → score
    [N, 2]."""
    raise NotImplementedError("TODO: vp_stats 본문을 구현하세요.")


def perturb(x0, t, z):
    """정답과 동일한 공개 signature를 유지한다. shape 계약: point xt [N, 2] + time [N, 1] → score
    [N, 2]."""
    raise NotImplementedError("TODO: perturb 본문을 구현하세요.")


# TODO: beta(t)=.1+9.9t VP marginal의 mean coefficient/std와 perturbed
# sample/target score를 구현하세요.
raise NotImplementedError

### 단계 3. 학습·업데이트

- **논문의 어느 부분인가:** `§3.3, Eq. (7)` — **continuous denoising score matching**
- **구현 이유:** perturbation kernel의 conditional score를 회귀한다.
- **읽을 코드:** 아래 `training` 셀에서 중간 tensor의 shape, gradient가 흐르는
  경로와 `detach` 또는 `no_grad` 경계를 확인합니다.

실행 결과만 보지 말고, 이 단계의 입력과 출력이 앞선 수식의 어떤 기호인지 먼저
주석으로 적은 뒤 실습하세요.
- **완료 증거:** 아래 셀의 `assert`와 출력 metric이 shape·loss 계약을 검사합니다.

In [ ]:
# TODO: [x_t,t] -> score MLP를 Eq. (7) weighted MSE로 220회 학습하세요.

# 통합 완료 조건: ScoreSdeLab.update(...)를 반복하고,
# history, parameter_delta, 유한 loss를 검증합니다.
raise NotImplementedError

### 단계 4. 평가·시각화

        - **논문의 어느 부분인가:** `§3.1 Eq. (5); §3.2 Eq. (6)` — **forward and reverse SDE**
        - **구현 이유:** forward drift/diffusion과 score가 들어간 reverse drift를 연결한다.
        - **읽을 코드:** 아래 `evaluation` 셀에서 중간 tensor의 shape, gradient가 흐르는
          경로와 `detach` 또는 `no_grad` 경계를 확인합니다.

        실행 결과만 보지 말고, 이 단계의 입력과 출력이 앞선 수식의 어떤 기호인지 먼저
        주석으로 적은 뒤 실습하세요.
        - **완료 증거:** 아래 셀의 `assert`와 출력 metric이 shape·loss 계약을 검사합니다.

- **평가 범위:** 이 값은 작은 학습 batch의 메커니즘 진단이며, held-out 생성 품질이나 원 논문의 benchmark 성능으로 해석하지 않습니다.

In [ ]:
# TODO: ScoreSdeLab.evaluate(...)를 호출해 portfolio_metrics를 만들고 시각화하세요.
raise NotImplementedError

reverse SDE에서 score가 필요한 이유는 무엇인가요?

### 단계 5. 통합 클래스가 실제 update와 evaluate를 소유하는지 검증

- **논문 위치:** `§3.1 Eq. (5); §3.2 Eq. (6)`의 **forward and reverse SDE**
- **핵심 식:**

  $$
  \mathcal L=\mathbb E_{t,x_0,x_t}\lambda(t)\lVert s_\theta(x_t,t)-\nabla_{x_t}\log p_{0t}(x_t|x_0)\rVert_2^2
  $$

- **입출력 shape:** `point xt [N, 2] + time [N, 1] → score [N, 2]`
- **구현 이유:** 앞 셀은 수식을 작은 연산으로 분해해 확인하고,
  이 셀은 같은 구성 요소를 `ScoreSdeLab`에 주입해
  최종 학습·평가 경로를 하나로 묶습니다. forward drift/diffusion과 score가 들어간 reverse drift를 연결한다.
- **완료 증거:** `update(...)`가 유한한 loss를 반환하고,
  `parameter_delta > 0`이며, `evaluate(...)`의 논문 대응 지표가 유한해야 합니다.
  `config()`만 호출하는 것은 실행 재현으로 보지 않습니다.

In [ ]:
# TODO 5: ScoreSdeLab 기반 학습의 최종 증거를 검증하세요.
# 계약: history, portfolio_metrics, parameter_delta를 모두 검사합니다.
raise NotImplementedError

## 실험 기록

1. 핵심 수식의 기호와 코드 변수 이름을 대응시키세요.
2. 어떤 assertion이 가장 중요한 구현 오류를 잡는지 설명하세요.
3. 이 미니 데이터로 검증할 수 없는 원 논문의 주장을 하나 적으세요.
4. seed를 유지하고 변수 하나만 바꾼 뒤 metric 전후를 기록하세요.